# ForecastEx

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Live markets

In [3]:
from get_live_markets import get_live_markets
markets = get_live_markets()

In [ ]:
from add_category_to_markets import add_category_to_markets
df_markets = add_category_to_markets(markets)

In [8]:
df_markets = df_markets[['category', 'name', 'symbol', 'conid']].sort_values(by=['category', 'name'])

In [12]:
df_markets[df_markets.symbol == 'MNYCG']

,category,name,symbol,conid
0,Election,General Election for New York City Mayor,MNYCG,796056051


In [14]:
conid_to_market = {market['conid']: market for market in markets}

In [17]:
market = conid_to_market[796056051]
market

{'symbol': 'MNYCG',
 'conid': 796056051,
 'name': 'General Election for New York City Mayor'}

## Get each contract of a market

In [18]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [19]:
conid = market['conid']

In [31]:
contracts = {}

In [20]:
contracts[conid] = fetch_all_contracts(conid)

In [23]:
import pandas as pd

In [96]:
submarkets = [{x: market[x] for x in ['symbol', 'conid', 'name']} for _ in contracts[conid]]

In [97]:
for contract, submarket in zip(contracts[conid], submarkets):
    for x in ['longDescription', 'expiration', 'conid', 'popularityRank', 'strikeLabel']:
        submarket[x] = contract[x]
    submarket['yesNo'] = 'NO' if contract['putOrCall'] == 'P' else 'YES'

In [98]:
pd.DataFrame(submarkets)

,symbol,conid,name,longDescription,expiration,popularityRank,strikeLabel,yesNo
0,MNYCG,796056496,General Election for New York City Mayor,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,1032416,Sliwa,YES
1,MNYCG,796056501,General Election for New York City Mayor,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,1032416,Sliwa,NO
2,MNYCG,796056506,General Election for New York City Mayor,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,1032416,Cuomo,YES
3,MNYCG,796056511,General Election for New York City Mayor,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,1032416,Cuomo,NO
4,MNYCG,796056514,General Election for New York City Mayor,Will Jim Walden win the New York City general election for mayor in 2025?,20251129,1032416,Walden,YES
5,MNYCG,796056519,General Election for New York City Mayor,Will Jim Walden win the New York City general election for mayor in 2025?,20251129,1032416,Walden,NO
6,MNYCG,796056520,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,1032416,Mamdani,YES
7,MNYCG,796056525,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,1032416,Mamdani,NO
8,MNYCG,796056531,General Election for New York City Mayor,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,1032416,Adams,YES
9,MNYCG,796056534,General Election for New York City Mayor,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,1032416,Adams,NO


## Get current probability for each market contract

In [99]:
from get_candidate_probability import get_candidate_probability

In [100]:
from tqdm.auto import tqdm

In [101]:
conid_to_probability = {}
for contract in tqdm(contracts[conid]):
    subconid = contract['conid']
    conid_to_probability[subconid] = get_candidate_probability(subconid)

  0%|          | 0/10 [00:00<?, ?it/s]

In [102]:
submarket['conid']

796056534

In [103]:
for submarket in submarkets:
    try:
        submarket['probability_pct'] = conid_to_probability[submarket['conid']]['probability_pct']
        submarket['volume'] = sum(conid_to_probability[submarket['conid']]['volume'])
    except:
        submarket['probability_pct'] = None
        submarket['volume'] = None

In [104]:
live_submarkets = [sm for sm in submarkets if sm['probability_pct'] is not None]

In [105]:
df = pd.DataFrame(live_submarkets)

In [106]:
# Separate YES and NO rows
df_yes = df[df['yesNo'] == 'YES'].copy()
df_no = df[df['yesNo'] == 'NO'].copy()

# Rename columns in YES dataframe
df_yes = df_yes.rename(columns={
    'conid': 'YES_conid',
    'probability_pct': 'YES_probability_pct'
})

# Rename columns in NO dataframe
df_no = df_no.rename(columns={
    'conid': 'NO_conid',
    'probability_pct': 'NO_probability_pct'
})

# Columns to use to join YES and NO dataframes 
join_cols = ['symbol', 'expiration', 'popularityRank', 'volume']

# Select columns for merging
df_yes_sel = df_yes[join_cols + ['YES_conid', 'YES_probability_pct']]
df_no_sel = df_no[join_cols + ['NO_conid', 'NO_probability_pct']]

# Merge on joining columns
df_merged = pd.merge(df_yes_sel, df_no_sel, on=join_cols, how='inner')

# If you like, you can also keep other YES columns like name, shortDescription, strikeLabel.
# For example, including those:
optional_cols = ['name', 'longDescription', 'strikeLabel']
df_merged = pd.merge(df_merged, df_yes[join_cols + optional_cols], on=join_cols, how='left')

# Drop duplicates if any appeared from the merge
df_merged = df_merged.drop_duplicates()

In [109]:
df_merged['spread'] = (df_merged.YES_probability_pct - df_merged.NO_probability_pct).abs()

In [110]:
df_merged

,symbol,expiration,popularityRank,volume,YES_conid,YES_probability_pct,NO_conid,NO_probability_pct,name,longDescription,strikeLabel,spread
0,MNYCG,20251129,1032416,46750.0,796056496,5.0,796056501,96.0,General Election for New York City Mayor,Will Curtis Sliwa win the New York City general election for mayor in 2025?,Sliwa,91.0
1,MNYCG,20251129,1032416,1939.0,796056506,17.0,796056511,84.0,General Election for New York City Mayor,Will Andrew Cuomo win the New York City general election for mayor in 2025?,Cuomo,67.0
2,MNYCG,20251129,1032416,49244.0,796056520,69.0,796056525,32.0,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,Mamdani,37.0
3,MNYCG,20251129,1032416,1615.0,796056531,7.0,796056534,94.0,General Election for New York City Mayor,Will Eric Adams win the New York City general election for mayor in 2025?,Adams,87.0


In [111]:
dfs = df_merged[df_merged.spread < 50]

In [112]:
dfs

,symbol,expiration,popularityRank,volume,YES_conid,YES_probability_pct,NO_conid,NO_probability_pct,name,longDescription,strikeLabel,spread
2,MNYCG,20251129,1032416,49244.0,796056520,69.0,796056525,32.0,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,Mamdani,37.0


In [130]:
s_conids = dfs.YES_conid.values.tolist()

In [131]:
s_conids

[796056520]

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [152]:
from run_get_OI import run_get_OI

In [153]:
conids_to_OI = await run_get_OI(s_conids)

WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056520: 1.18M
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None


In [154]:
conids_to_OI

{'796056520': '1.18M'}

In [155]:
def OI_to_integer(OI):
    if OI[-1] == 'K':
        return float(OI[:-1])*1000.0
    elif OI[-1] == 'M':
        return float(OI[:-1])*1000000.0
    else:
        return float(OI)

In [156]:
conid_to_OI = {int(conid): OI_to_integer(OI) for conid, OI in conids_to_OI.items()}

In [157]:
dfs = dfs.copy()

In [158]:
dfs['OI'] = dfs.YES_conid.apply(lambda x: conid_to_OI[x])

In [159]:
dfs

,symbol,expiration,popularityRank,volume,YES_conid,YES_probability_pct,NO_conid,NO_probability_pct,name,longDescription,strikeLabel,spread,OI
2,MNYCG,20251129,1032416,49244.0,796056520,69.0,796056525,32.0,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,Mamdani,37.0,1180000.0


In [160]:
dfs.columns

Index(['symbol', 'expiration', 'popularityRank', 'volume', 'YES_conid',
       'YES_probability_pct', 'NO_conid', 'NO_probability_pct', 'name',
       'longDescription', 'strikeLabel', 'spread', 'OI'],
      dtype='object')

In [161]:
dff = dfs[['expiration', 'longDescription', 'YES_probability_pct', 'NO_probability_pct', 'spread', 'OI', 'volume']]

In [151]:
dff

,expiration,longDescription,YES_probability_pct,NO_probability_pct,spread,OI,volume
2,20251129,Will Zohran Mamdani win the New York City general election for mayor in 2025?,69.0,32.0,37.0,1180000.0,49244.0
